# 07 - Ajuste del dataset a partir del análisis exploratorio

Parte del dataset unificado completo (35 embalses, 2000-2023) y aplica los descartes en orden de exigencia:
1. Condiciones indispensables (independientes del periodo de inicio)
2. Selección del conjunto experimental por cobertura de calidad
3. Decisión del recorte temporal, sobre los embalses supervivientes
4. Descarte de variables por cobertura, sobre el periodo definitivo

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append("..")
DIR_PROCESSED = Path("../data/processed")

TEST_INI, TEST_FIN = pd.Timestamp("2022-01-01"), pd.Timestamp("2023-12-31")
MIN_COBERTURA = 50

dataset = pd.read_parquet(DIR_PROCESSED / "dataset_unificado_completo.parquet")
print(f"Dataset completo: {dataset['ID_SAIH'].nunique()} embalses | "
      f"{dataset['fecha'].min():%Y-%m-%d} a {dataset['fecha'].max():%Y-%m-%d} | {len(dataset):,} filas")
embudo = {"inicial": dataset['ID_SAIH'].nunique()}

Dataset completo: 35 embalses | 2000-01-01 a 2023-12-31 | 306,810 filas


## 1. Condiciones indispensables

Se descartan los embalses que no cumplen las condiciones mínimas, por razones independientes del periodo de inicio: serie del objetivo sin dinámica, ausencia de objetivo en el periodo de test, o ausencia total de meteorología.

In [2]:
fuera = {}

# E16D: serie plana (central hidroeléctrica, no embalse regulador)
for emb, g in dataset.groupby("ID_SAIH"):
    rango = g["pct_llenado"].max() - g["pct_llenado"].min()
    if rango < 1:  # prácticamente constante
        fuera[emb] = "serie plana"

# Sin objetivo en el periodo de test (2022-2023)
for emb, g in dataset.groupby("ID_SAIH"):
    test = g[(g["fecha"] >= TEST_INI) & (g["fecha"] <= TEST_FIN)]
    if test["pct_llenado"].notna().sum() == 0:
        fuera[emb] = fuera.get(emb, "sin objetivo en test")

# Sin ninguna meteo AEMET en absoluto (en todo el periodo)
meteo_base = ["aemet_temp_media_c", "aemet_precipitacion_mm", "aemet_humedad_pct"]
for emb, g in dataset.groupby("ID_SAIH"):
    if g[meteo_base].notna().any(axis=1).sum() == 0:
        fuera[emb] = fuera.get(emb, "sin meteo AEMET")

print("Descartados por condiciones indispensables:")
for emb, motivo in sorted(fuera.items()):
    print(f"  {emb}: {motivo}")

dataset = dataset[~dataset["ID_SAIH"].isin(fuera)].copy()
embudo["tras_indispensables"] = dataset["ID_SAIH"].nunique()
print(f"\nEmbalses restantes: {embudo['tras_indispensables']}")

Descartados por condiciones indispensables:
  E024: sin objetivo en test
  E16D: serie plana

Embalses restantes: 33


## 2. Selección del conjunto experimental por cobertura de calidad

El experimento compara escenarios con y sin variables de calidad, por lo que solo son útiles los embalses con datos de calidad suficientes. Se mide la cobertura sobre el histórico disponible de cada embalse y sobre el periodo de test, sin presuponer aún el recorte del periodo de inicio.

In [3]:
cal_cols = [c for c in dataset.columns if c.startswith("cal_")]
dataset["tiene_calidad"] = dataset[cal_cols].notna().any(axis=1)

res = []
for emb, g in dataset.groupby("ID_SAIH"):
    hist = g["tiene_calidad"].mean() * 100
    test = g[(g["fecha"] >= TEST_INI) & (g["fecha"] <= TEST_FIN)]["tiene_calidad"].mean() * 100
    res.append({"ID_SAIH": emb, "cob_historico": round(hist, 1), "cob_test": round(test, 1)})
res = pd.DataFrame(res).sort_values("cob_test")
print(res.to_string(index=False))

experimentales = sorted(res.query(
    "cob_historico >= @MIN_COBERTURA and cob_test >= @MIN_COBERTURA")["ID_SAIH"])
print(f"\nConjunto experimental: {len(experimentales)} embalses")
print(experimentales)

dataset = dataset[dataset["ID_SAIH"].isin(experimentales)].copy()
embudo["tras_calidad"] = dataset["ID_SAIH"].nunique()

ID_SAIH  cob_historico  cob_test
   E003            0.0       0.0
   E014            0.0       0.0
   E017            0.0       0.0
   E022            0.0       0.0
   E021            0.0       0.0
   E020            0.0       0.0
   E015            0.0       0.0
   E05A            0.0       0.0
   E16A            0.0       0.0
   E18A            0.0       0.0
   E19A            0.0       0.0
   E36A            0.0       0.0
   E023           55.1      19.3
   E013           55.1      19.3
   E350           55.1      19.3
   E35A           86.7      96.0
   E028           88.7      98.9
   E027           81.8      99.5
   E031           81.8      99.5
   E030           81.8      99.5
   E026           81.8      99.5
   E025           81.8      99.5
   E002           81.8      99.5
   E009           82.2     100.0
   E008           82.2     100.0
   E001           15.8     100.0
   E011           86.9     100.0
   E029           90.7     100.0
   E07A           82.2     100.0
   E033   

## 3. Decisión del recorte temporal

Sobre los embalses supervivientes se examina la completitud de las variables clave antes y después de 2006, para fundamentar si procede acotar el periodo de estudio.

In [4]:
# Diagnóstico: cobertura anual de las variables clave sobre los 17 experimentales
df_diag = pd.read_parquet(DIR_PROCESSED / "dataset_unificado_completo.parquet")
df_diag = df_diag[df_diag["ID_SAIH"].isin(experimentales)].copy()
df_diag["anio"] = df_diag["fecha"].dt.year

vars_ver = ["aportacion_m3s", "aemet_temp_media_c", "aemet_humedad_pct",
            "aemet_viento_ms", "aemet_insolacion_h", "aemet_presion_max_hpa",
            "cal_abajo_ph", "cal_arriba_ph"]

cob_anual = (df_diag.groupby("anio")[vars_ver]
             .apply(lambda g: g.notna().mean() * 100).round(0).astype(int))
print("Cobertura (%) por año sobre los 17 experimentales:\n")
print(cob_anual.to_string())

Cobertura (%) por año sobre los 17 experimentales:

      aportacion_m3s  aemet_temp_media_c  aemet_humedad_pct  aemet_viento_ms  aemet_insolacion_h  aemet_presion_max_hpa  cal_abajo_ph  cal_arriba_ph
anio                                                                                                                                                
2000              65                  86                 72               67                  29                     66             0              0
2001              65                  87                 73               68                  29                     67            34             24
2002              65                  91                 75               67                  51                     69            62             36
2003              65                  85                 63               42                  40                     60            30             15
2004              65                  90              

In [5]:
# --- Diagnóstico: embalses sin aportación antes de 2006 (sobre los 17 experimentales) ---
CORTE = pd.Timestamp("2006-01-01")

# Recargar en el estado "17 experimentales, periodo completo" para no depender
# de si 'dataset' ya está recortado en este punto del notebook
_diag = pd.read_parquet(DIR_PROCESSED / "dataset_unificado_completo.parquet")
_diag = _diag[_diag["ID_SAIH"].isin(experimentales)].copy()

# Cobertura de aportación de cada embalse en el tramo anterior a 2006
_pre = _diag[_diag["fecha"] < CORTE]
cob_apor_pre = (_pre.groupby("ID_SAIH")["aportacion_m3s"]
                .apply(lambda s: s.notna().mean() * 100)
                .round(1)
                .sort_values())

sin_aportacion = cob_apor_pre[cob_apor_pre == 0]
print(f"Embalses experimentales sin aportación alguna antes de 2006: {len(sin_aportacion)}")
print(f"  {list(sin_aportacion.index)}\n")
print("Cobertura de aportación pre-2006 por embalse (%):")
print(cob_apor_pre.to_string())

Embalses experimentales sin aportación alguna antes de 2006: 5
  ['E011', 'E025', 'E026', 'E570', 'E571']

Cobertura de aportación pre-2006 por embalse (%):
ID_SAIH
E011      0.0
E025      0.0
E026      0.0
E570      0.0
E571      0.0
E028     15.5
E002    100.0
E009    100.0
E027    100.0
E029    100.0
E031    100.0
E030    100.0
E033    100.0
E07A    100.0
E32A    100.0
E008    100.0
E35A    100.0


In [6]:
TRAIN_FIN = pd.Timestamp("2021-12-31")
vars_clave = ["pct_llenado", "aportacion_m3s", "salida_m3s",
              "aemet_temp_media_c", "aemet_precipitacion_mm", "aemet_humedad_pct"] + cal_cols

# Comparar cobertura sobre el TRAIN según empiece en 2000 o en 2006
train_completo = dataset[dataset["fecha"] <= TRAIN_FIN]              # 2000-2021
train_recortado = dataset[(dataset["fecha"] >= CORTE) & (dataset["fecha"] <= TRAIN_FIN)]  # 2006-2021

print("Efecto del recorte sobre la cobertura del train (17 embalses):\n")
print(f"{'variable':<32} {'2000-2021':>10} {'2006-2021':>10} {'mejora':>8}")
for v in vars_clave:
    c0 = train_completo[v].notna().mean() * 100
    c6 = train_recortado[v].notna().mean() * 100
    print(f"{v:<32} {c0:>9.1f}% {c6:>9.1f}% {c6-c0:>+7.1f}")

# Cuántas filas cuesta cada opción
print("\n--- Coste en datos de cada opción ---")
sin_apor_pre2006 = ["E011", "E025", "E026", "E570", "E571"]  # ajustar según celda anterior
n_recorte = len(dataset[dataset["fecha"] < CORTE])
n_embalses = len(dataset[dataset["ID_SAIH"].isin(sin_apor_pre2006)])
print(f"Opción A (recortar 2000-2005): se pierden {n_recorte:,} filas")
print(f"Opción B (quitar {len(sin_apor_pre2006)} embalses sin aportación): se pierden {n_embalses:,} filas")

Efecto del recorte sobre la cobertura del train (17 embalses):

variable                          2000-2021  2006-2021   mejora
pct_llenado                           99.8%      99.7%    -0.0
aportacion_m3s                        90.3%      99.6%    +9.3
salida_m3s                            99.5%      99.7%    +0.2
aemet_temp_media_c                    94.8%      97.4%    +2.6
aemet_precipitacion_mm                93.9%      96.2%    +2.3
aemet_humedad_pct                     88.3%      94.5%    +6.2
cal_arriba_amonio_mgl                 30.1%      33.1%    +3.0
cal_arriba_conductividad_uscm         34.0%      37.7%    +3.7
cal_arriba_fosfatos_mgl                3.3%       4.6%    +1.2
cal_arriba_materia_organica_m1         6.1%       8.4%    +2.3
cal_arriba_oxigeno_mgl                33.8%      37.5%    +3.7
cal_arriba_ph                         34.2%      37.9%    +3.8
cal_arriba_temp_agua_c                34.2%      38.0%    +3.8
cal_arriba_turbidez_ntu               33.3%      36.8

In [7]:
# A la vista del análisis anterior, se acota el periodo a 2006-2023
FECHA_INICIO = pd.Timestamp("2006-01-01")
FECHA_FIN = pd.Timestamp("2023-12-31")

dataset = dataset[(dataset["fecha"] >= FECHA_INICIO) & (dataset["fecha"] <= FECHA_FIN)].copy()
embudo["periodo"] = f"{dataset['fecha'].min():%Y} a {dataset['fecha'].max():%Y}"
print(f"Periodo: {embudo['periodo']} | Filas: {len(dataset):,}")

Periodo: 2006 a 2023 | Filas: 111,758


## 4. Descarte de variables por cobertura sobre el periodo definitivo

Con el periodo ya fijado, se evalúa la completitud de cada variable y se descartan las de cobertura insuficiente o ausentes en varios embalses.

In [8]:
# Cobertura de cada variable sobre el periodo definitivo
print("Cobertura por variable (periodo definitivo) y embalses sin dato:\n")
candidatas = [c for c in dataset.columns
              if c not in ["fecha", "ID_SAIH", "Nombre_SAIH", "Sistema",
                           "Capacidad_hm3", "tiene_calidad"]]
for c in candidatas:
    cob = dataset[c].notna().mean() * 100
    sin = dataset.groupby("ID_SAIH")[c].apply(lambda s: s.notna().sum() == 0).sum()
    marca = "  <-- candidata a descarte" if (cob < 70 or sin > 2) else ""
    print(f"  {c:<32} {cob:5.1f}%  sin dato en {sin} embalses{marca}")

Cobertura por variable (periodo definitivo) y embalses sin dato:

  volumen_hm3                       99.7%  sin dato en 0 embalses
  pct_llenado                       99.7%  sin dato en 0 embalses
  aportacion_m3s                    99.6%  sin dato en 0 embalses
  salida_m3s                        99.7%  sin dato en 0 embalses
  aemet_temp_media_c                97.7%  sin dato en 0 embalses
  aemet_temp_min_c                  97.7%  sin dato en 0 embalses
  aemet_temp_max_c                  97.7%  sin dato en 0 embalses
  aemet_precipitacion_mm            96.5%  sin dato en 0 embalses
  aemet_humedad_pct                 95.1%  sin dato en 0 embalses
  aemet_viento_ms                   79.9%  sin dato en 2 embalses
  aemet_insolacion_h                61.8%  sin dato en 5 embalses  <-- candidata a descarte
  aemet_presion_max_hpa             63.9%  sin dato en 5 embalses  <-- candidata a descarte
  aemet_presion_min_hpa             63.9%  sin dato en 5 embalses  <-- candidata a descart

In [9]:
# Identificar los embalses sin viento en el periodo definitivo
df_v = pd.read_parquet(DIR_PROCESSED / "dataset_unificado_completo.parquet")
df_v = df_v[(df_v["ID_SAIH"].isin(experimentales)) &
            (df_v["fecha"] >= pd.Timestamp("2006-01-01"))].copy()

print("Cobertura de viento por embalse (2006-2023):")
cob_viento = (df_v.groupby(["ID_SAIH"])["aemet_viento_ms"]
              .apply(lambda s: round(s.notna().mean() * 100, 1))
              .sort_values())
# añadir nombre
nombres = df_v.groupby("ID_SAIH")["Nombre_SAIH"].first()
for emb, cob in cob_viento.items():
    print(f"  {emb} ({nombres[emb]}): {cob}%")

Cobertura de viento por embalse (2006-2023):
  E35A (Conchas - Presa): 0.0%
  E571 (Pumares): 0.0%
  E033 (Frieira): 14.1%
  E028 (Vilasouto): 76.6%
  E570 (Santiago): 89.4%
  E026 (Edrada Mao): 96.5%
  E025 (Leboreiro Mao): 96.5%
  E32A (Albarellos - Presa): 96.9%
  E029 (San Pedro): 97.9%
  E027 (San Esteban): 97.9%
  E002 (Os Peares): 97.9%
  E011 (Peñarrubia): 98.9%
  E009 (Montearenas): 98.9%
  E07A (Presa de Bárcena): 98.9%
  E008 (Fuente del Azufre): 98.9%
  E031 (Castrelo): 99.9%
  E030 (Velle): 99.9%


In [10]:
# Verificar cobertura del bloque único de calidad (arriba con prioridad, abajo en defecto)
df_cal = pd.read_parquet(DIR_PROCESSED / "dataset_unificado_completo.parquet")
df_cal = df_cal[(df_cal["ID_SAIH"].isin(experimentales)) &
                (df_cal["fecha"] >= pd.Timestamp("2006-01-01"))].copy()

PARAMS = ["amonio_mgl", "conductividad_uscm", "oxigeno_mgl", "ph", "temp_agua_c", "turbidez_ntu"]

print("Cobertura sobre 2006-2023 (17 embalses):\n")
print(f"{'parámetro':<20} {'arriba':>8} {'abajo':>8} {'fusión':>8} {'emb.fusión':>11}")
for p in PARAMS:
    arr = df_cal[f"cal_arriba_{p}"]
    aba = df_cal[f"cal_abajo_{p}"]
    fus = arr.fillna(aba)
    c_arr = arr.notna().mean() * 100
    c_aba = aba.notna().mean() * 100
    c_fus = fus.notna().mean() * 100
    # embalses con al menos un dato en la fusión
    emb_fus = df_cal.assign(_f=fus).groupby("ID_SAIH")["_f"].apply(lambda s: s.notna().any()).sum()
    print(f"{p:<20} {c_arr:>7.1f}% {c_aba:>7.1f}% {c_fus:>7.1f}% {emb_fus:>8}/17")

Cobertura sobre 2006-2023 (17 embalses):

parámetro              arriba    abajo   fusión  emb.fusión
amonio_mgl              33.8%    62.3%    82.3%       17/17
conductividad_uscm      38.0%    71.6%    92.8%       17/17
oxigeno_mgl             37.8%    70.6%    92.2%       17/17
ph                      38.1%    71.4%    92.8%       17/17
temp_agua_c             38.2%    71.7%    93.1%       17/17
turbidez_ntu            36.8%    69.1%    89.6%       17/17


In [11]:
# --- Paso 4: fusión de calidad en bloque único y descarte de variables ---

# 4a. Bloque único de calidad: arriba con prioridad, abajo en defecto
PARAMS_CAL = ["amonio_mgl", "conductividad_uscm", "oxigeno_mgl",
              "ph", "temp_agua_c", "turbidez_ntu"]
for p in PARAMS_CAL:
    dataset[f"cal_{p}"] = dataset[f"cal_arriba_{p}"].fillna(dataset[f"cal_abajo_{p}"])

# 4b. Eliminar columnas arriba/abajo originales (incluye fosfatos y materia orgánica),
#     las meteo descartadas y la auxiliar tiene_calidad
cols_cal_orig = [c for c in dataset.columns
                 if c.startswith("cal_arriba_") or c.startswith("cal_abajo_")]
meteo_fuera = ["aemet_viento_ms", "aemet_insolacion_h",
               "aemet_presion_max_hpa", "aemet_presion_min_hpa"]
a_quitar = cols_cal_orig + meteo_fuera + ["tiene_calidad"]
a_quitar = [c for c in a_quitar if c in dataset.columns]

dataset = dataset.drop(columns=a_quitar)

print(f"Calidad fusionada en bloque único: {[f'cal_{p}' for p in PARAMS_CAL]}")
print(f"Meteo descartada: {meteo_fuera}")
print(f"Columnas finales ({len(dataset.columns)}): {sorted(dataset.columns)}")

Calidad fusionada en bloque único: ['cal_amonio_mgl', 'cal_conductividad_uscm', 'cal_oxigeno_mgl', 'cal_ph', 'cal_temp_agua_c', 'cal_turbidez_ntu']
Meteo descartada: ['aemet_viento_ms', 'aemet_insolacion_h', 'aemet_presion_max_hpa', 'aemet_presion_min_hpa']
Columnas finales (20): ['Capacidad_hm3', 'ID_SAIH', 'Nombre_SAIH', 'Sistema', 'aemet_humedad_pct', 'aemet_precipitacion_mm', 'aemet_temp_max_c', 'aemet_temp_media_c', 'aemet_temp_min_c', 'aportacion_m3s', 'cal_amonio_mgl', 'cal_conductividad_uscm', 'cal_oxigeno_mgl', 'cal_ph', 'cal_temp_agua_c', 'cal_turbidez_ntu', 'fecha', 'pct_llenado', 'salida_m3s', 'volumen_hm3']


In [12]:
print("=== EMBUDO DE DESCARTES ===")
print(f"Inicial:             {embudo['inicial']} embalses")
print(f"Tras indispensables: {embudo['tras_indispensables']} embalses")
print(f"Tras calidad:        {embudo['tras_calidad']} embalses")
print(f"Periodo:             {embudo['periodo']}")
print(f"\nDataset ajustado: {len(dataset):,} filas x {len(dataset.columns)} columnas")

dataset.to_parquet(DIR_PROCESSED / "dataset_ajustado.parquet", index=False)
pd.DataFrame({"ID_SAIH": experimentales}).to_parquet(
    DIR_PROCESSED / "embalses_experimento_calidad.parquet", index=False)
print("Guardado: dataset_ajustado.parquet y embalses_experimento_calidad.parquet")

=== EMBUDO DE DESCARTES ===
Inicial:             35 embalses
Tras indispensables: 33 embalses
Tras calidad:        17 embalses
Periodo:             2006 a 2023

Dataset ajustado: 111,758 filas x 20 columnas
Guardado: dataset_ajustado.parquet y embalses_experimento_calidad.parquet
